In [18]:
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# --- Build ONE common table from telemetry + endurance (keeps your column names) ---

import pandas as pd, numpy as np
from pathlib import Path

# 1) Endurance normalize (cumulative race time + sector split) --------------------
def _parse_elapsed(x):
    if pd.isna(x): return np.nan
    s=str(x); p=s.split(":")
    try:
        if len(p)==2:  # M:SS.mmm
            m,sec=p; return int(m)*60+float(sec)
        if len(p)==3:  # H:MM:SS.mmm
            h,m,sec=p; return int(h)*3600+int(m)*60+float(sec)
    except: return np.nan
    return np.nan

def normalize_endurance(df_endurance: pd.DataFrame) -> pd.DataFrame:
    e = df_endurance.copy()
    e["meta_session"]   = e["meta_session"].astype(str)
    e["vehicle_number"] = e["vehicle_number"].astype(str)

    # lap time seconds
    if "lap_time_s" in e.columns:
        e["lapTime_s"] = pd.to_numeric(e["lap_time_s"], errors="coerce")
    elif "lap_time" in e.columns:
        e["lapTime_s"] = e["lap_time"].apply(_parse_elapsed)
    else:
        raise ValueError("Endurance needs 'lap_time_s' or 'lap_time'.")

    # cumulative at lap end (prefer 'elapsed' if present)
    if "elapsed" in e.columns:
        e["cumulative_s"] = e["elapsed"].apply(_parse_elapsed)
    else:
        e = e.sort_values(["meta_session","vehicle_number","lap"])
        e["cumulative_s"] = e.groupby(["meta_session","vehicle_number"])["lapTime_s"].cumsum()

    e["lap_start_s"] = e["cumulative_s"] - e["lapTime_s"]

    # sector boundaries (use s1/s2/s3 if present, else equal thirds)
    thirds = e["lapTime_s"]/3.0
    s1 = pd.to_numeric(e.get("s1_seconds", np.nan), errors="coerce")
    s2 = pd.to_numeric(e.get("s2_seconds", np.nan), errors="coerce")
    s3 = pd.to_numeric(e.get("s3_seconds", np.nan), errors="coerce")
    e["t_s1"] = np.where(s1.notna(), s1, thirds)
    e["t_s2"] = np.where(s2.notna(), e["t_s1"] + s2, 2*thirds)
    e["t_s3"] = np.where(s3.notna(), e["lapTime_s"], e["lapTime_s"])

    keep = ["meta_session","vehicle_number","lap","lap_start_s","cumulative_s","lapTime_s","t_s1","t_s2","t_s3"]
    return e[keep].sort_values(["meta_session","vehicle_number","lap"])

# 2) Telemetry → regular time grid (no renaming) ---------------------------------
def telemetry_to_grid(df_tel: pd.DataFrame, rate_hz: int = 10) -> pd.DataFrame:
    """
    Accepts either:
      - long telemetry with columns: meta_session, vehicle_number, meta_time, telemetry_name, telemetry_value
      - wide telemetry already exploded at meta_time
    Produces a uniform grid per (meta_session, vehicle_number) with:
      meta_session, vehicle_number, meta_time, t_s (car-relative), t_rel (session-relative), and all telemetry columns kept.
    """
    df = df_tel.copy()
    df["meta_session"]   = df["meta_session"].astype(str)
    df["vehicle_number"] = df["vehicle_number"].astype(str)
    df["meta_time"]      = pd.to_datetime(df["meta_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["meta_time"])

    # If long → pivot at timestamp level
    if {"telemetry_name","telemetry_value"}.issubset(df.columns):
        df["telemetry_value"] = pd.to_numeric(df["telemetry_value"], errors="coerce")
        df = (df.pivot_table(index=["meta_session","vehicle_number","meta_time"],
                             columns="telemetry_name", values="telemetry_value", aggfunc="mean")
                .reset_index())
        df.columns.name = None

    # --- RAW PATH: keep original timestamps when rate_hz is None ---
    if rate_hz is None:
        parts = []
        session_starts = df.groupby("meta_session")["meta_time"].min().rename("session_start_ts")
        for (ms, vid), g in df.groupby(["meta_session","vehicle_number"], sort=False):
            g = g.sort_values("meta_time")
            t0 = g["meta_time"].iloc[0]
            ss = session_starts.loc[ms]
            g = g.assign(
                t_s   =(g["meta_time"]-t0).dt.total_seconds().astype("float32"),
                t_rel =(g["meta_time"]-ss).dt.total_seconds().astype("float32")
            )
            parts.append(g)
        return (pd.concat(parts, ignore_index=True)
                .sort_values(["meta_session","vehicle_number","meta_time"]))


    # Regularize to uniform grid per group (e.g., 10 Hz)
    rule = f"{int(1000 // rate_hz)}L"  # '100L' for 10 Hz
    parts = []
    session_starts = df.groupby("meta_session")["meta_time"].min().rename("session_start_ts")

    metric_cols = [c for c in df.columns if c not in ["meta_session","vehicle_number","meta_time"]]

    for (ms, vid), g in df.groupby(["meta_session","vehicle_number"], sort=False):
        g = g.sort_values("meta_time").set_index("meta_time")
        r = g[metric_cols].resample(rule).mean()
        # limit fill to avoid drawing chords across big gaps
        r = r.ffill(limit=rate_hz//2).bfill(limit=1)  # ~0.5 s ffill max
        r = r.reset_index()
        r.insert(0, "meta_session", ms)
        r.insert(1, "vehicle_number", vid)
        # times
        t0 = r["meta_time"].min()
        ss = session_starts.loc[ms]
        r["t_s"]   = (r["meta_time"] - t0).dt.total_seconds().astype("float32")
        r["t_rel"] = (r["meta_time"] - ss).dt.total_seconds().astype("float32")
        parts.append(r)

    grid = pd.concat(parts, ignore_index=True).sort_values(["meta_session","vehicle_number","meta_time"])
    return grid

# 3) Attach lap/sector & gap-to-leader (keeps original cols) ---------------------
def attach_lap_sector_and_gaps(grid: pd.DataFrame, e_norm: pd.DataFrame, gap_bin_ms: int = 500) -> pd.DataFrame:
    out = []

    # Ensure numeric, matching dtypes for merge_asof keys
    grid = grid.copy()
    e_norm = e_norm.copy()
    grid["t_s"] = pd.to_numeric(grid["t_s"], errors="coerce").astype("float64")
    e_norm["lap_start_s"] = pd.to_numeric(e_norm["lap_start_s"], errors="coerce").astype("float64")

    for (ms, vid), g in grid.groupby(["meta_session","vehicle_number"], sort=False):
        ee = e_norm[(e_norm.meta_session==ms) & (e_norm.vehicle_number==vid)]
        if ee.empty or g.empty:
            out.append(g.assign(lap=np.nan, sector=np.nan, sector_progress=np.nan, run_cum_s=np.nan))
            continue

        g_sorted  = g.sort_values("t_s")
        ee_sorted = ee.sort_values("lap_start_s")

        m = pd.merge_asof(
            g_sorted,
            ee_sorted[["lap","lap_start_s","lapTime_s","t_s1","t_s2","t_s3"]],
            left_on="t_s", right_on="lap_start_s",
            direction="backward", allow_exact_matches=True
        )

        el = m["t_s"] - m["lap_start_s"]
        conds = [el <= m["t_s1"], el <= m["t_s2"], el <= m["t_s3"]]
        m["sector"] = np.select(conds, [1,2,3], default=np.nan)

        seg_len = np.select(
            [m["sector"].eq(1), m["sector"].eq(2), m["sector"].eq(3)],
            [m["t_s1"], m["t_s2"]-m["t_s1"], m["t_s3"]-m["t_s2"]],
            default=np.nan
        )
        seg_start = np.select(
            [m["sector"].eq(1), m["sector"].eq(2), m["sector"].eq(3)],
            [0, m["t_s1"], m["t_s2"]],
            default=np.nan
        )
        m["sector_progress"] = np.clip((el - seg_start)/seg_len, 0, 1)
        m["run_cum_s"] = m["lap_start_s"] + np.clip(el, 0, m["lapTime_s"])
        out.append(m)

    uni = pd.concat(out, ignore_index=True)

    # Gaps on a common session clock
    step = gap_bin_ms / 1000.0
    uni["_tbin"] = (uni["t_rel"] / step).round().astype("int32")
    leader = (uni.dropna(subset=["run_cum_s"])
                .groupby(["meta_session","_tbin"])["run_cum_s"].min()
                .rename("leader_run_cum_s")).reset_index()
    uni = uni.merge(leader, on=["meta_session","_tbin"], how="left")
    uni["gap_s"] = uni["run_cum_s"] - uni["leader_run_cum_s"]
    return uni.drop(columns=["_tbin"])

# 4) Orchestrator ---------------------------------------------------------------
def build_common_table(df_tel: pd.DataFrame, df_endurance: pd.DataFrame,
                       rate_hz: int = 10, gap_bin_ms: int = 500,
                       out_path: str | None = None) -> pd.DataFrame:
    e_norm = normalize_endurance(df_endurance)
    grid   = telemetry_to_grid(df_tel, rate_hz=rate_hz)
    uni    = attach_lap_sector_and_gaps(grid, e_norm, gap_bin_ms=gap_bin_ms)



    # STEP TO REMOVE CARS WHICH DON'T HAVE ANY ENDURANCE DATA - 0, 16 and 78
    cars_to_omit = ['0', '16', '78']
    print(uni[~uni['vehicle_number'].isin(cars_to_omit)]['lap'])
    uni = uni[~uni['vehicle_number'].isin(cars_to_omit)]



    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        if out_path.endswith(".csv"):
            uni.to_csv(out_path, index=False)
        else:
            if not out_path.endswith(".parquet"):
                out_path = out_path + ".parquet"
            uni.to_parquet(out_path, index=False)
    return uni


In [40]:
tele_wide = pd.read_parquet('simulation-data/telemetry_wide.parquet')
df_endurance = pd.read_csv("simulation-data/endurance.csv")

In [54]:
# common = build_common_table(tele_wide, df_endurance, rate_hz=10, gap_bin_ms=500, out_path="race_unified.parquet")

common = build_common_table(tele_wide, df_endurance, rate_hz=None, gap_bin_ms=250, out_path="race_unified.parquet")


4213       1.0
4214       1.0
4215       1.0
4216       1.0
4217       1.0
          ... 
197078    27.0
197079    27.0
197080    27.0
197081    27.0
197082    27.0
Name: lap, Length: 165671, dtype: float64


In [55]:
common.shape

(165671, 28)

In [56]:
common.to_parquet('simulation-data/common.parquet', index=False)